In [32]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import precision_score, recall_score, f1_score, confusion_matrix
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier
from sklearn.naive_bayes import BernoulliNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score
import matplotlib.pyplot as plt
import seaborn as sns


In [ ]:
# load combined dataset 
df = pd.read_csv('combined_dataset.csv')
df.head()
# classes 1 indicates the presence of malware and 0 indicates benign

,SEND_SMS,RECEIVE_SMS,HttpPost.init,android.intent.action.PACKAGE_REMOVED,BLUETOOTH,android.intent.action.BOOT_COMPLETED,READ_PHONE_STATE,android.intent.action.SEND,android.intent.action.SEND_MULTIPLE,.system.app,...,IBinder,Ljava.lang.Class.cast,Ljava.lang.Class.getMethod,READ_PROFILE,HttpUriRequest,android.os.IBinder,android.content.pm.PackageInfo,Binder,READ_EXTERNAL_STORAGE,class
0,1,1,0,0,0,1,1,0,0,0,...,1,0,1,0,0,1,0,1,0,1
1,0,0,1,0,0,1,1,0,0,0,...,1,0,1,0,1,1,0,1,0,1
2,0,0,0,0,0,0,0,0,0,0,...,1,0,0,0,0,1,0,1,0,0
3,0,0,1,0,0,0,1,0,0,0,...,1,1,1,0,1,1,1,1,0,0
4,0,0,1,0,0,0,0,0,0,0,...,1,1,1,0,1,1,1,1,0,0


In [3]:
# dropping the class column
X = df.drop(columns=['class'])
# storing the class column
y = df['class']

In [7]:
# splitting the dataset into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

In [8]:
# train base classifiers
models = {
    "RandomForest": RandomForestClassifier(n_estimators=100, random_state=42),
    "AdaBoost": AdaBoostClassifier(n_estimators=100, random_state=42),
    "NaiveBayes": BernoulliNB(),
    "DecisionTree": DecisionTreeClassifier(random_state=42),
    "SVM": SVC(kernel='linear', probability=True)
}

In [ ]:
# training all models
for name, model in models.items():
    print(f"Training {name}...")
    model.fit(X_train,y_train)
    

Training RandomForest...
Training AdaBoost...
Training NaiveBayes...
Training DecisionTree...
Training SVM...


In [10]:
# getting prediction probabilities from base models

probs = []

for name, model in models.items():
    print(f"Getting prediction probabilities from {name}...")
    prob = model.predict_proba(X_test)[:, 1]  # get the probability of the positive class
    probs.append(prob)

Getting prediction probabilities from RandomForest...
Getting prediction probabilities from AdaBoost...
Getting prediction probabilities from NaiveBayes...
Getting prediction probabilities from DecisionTree...
Getting prediction probabilities from SVM...


In [11]:
# stacking the probabilities as columns: shape (n_samples, n_models)
prob_matrix = np.column_stack(probs)

In [15]:
# converting probabilities to rankings  (lower = higher confidence)
rank_matrix = prob_matrix.argsort(axis=1).argsort(axis=1)  #rank rows
avg_rank = rank_matrix.mean(axis=1)

In [21]:
# Average the predicted probabilities directly
avg_probs = prob_matrix.mean(axis=1)
final_preds = (avg_probs > 0.5).astype(int)  # standard threshold


In [ ]:
# evaluating the performance of the stacked model
precision = precision_score(y_test, final_preds)
recall = recall_score(y_test, final_preds)
f1 = f1_score(y_test, final_preds)
cm = confusion_matrix(y_test, final_preds)


print("📊 DroidFusion-Style Fusion Results:")
print(f"Precision: {precision:.4f}")
print(f"Recall (TPR): {recall:.4f}")
print(f"F1-score: {f1:.4f}")
print("Confusion Matrix:")
print(cm)

📊 DroidFusion-Style Fusion Results:
Precision: 0.9795
Recall (TPR): 0.9838
F1-score: 0.9817
Confusion Matrix:
[[2375   28]
 [  22 1340]]


In [23]:
print("Average ranks range:", avg_rank.min(), avg_rank.max())
print("Predicted malware count:", final_preds.sum())


Average ranks range: 2.0 2.0
Predicted malware count: 1368


In [33]:
# Implementing AAB implementation
# Splitting training into train + val to evaluate accuracies
X_train_main, X_val, y_train_main, y_val = train_test_split(X_train, y_train, test_size=0.2, stratify=y_train, random_state=42)

In [34]:

# Re-training base models on the smaller train split
model_weights = {}
for name, model in models.items():
    model.fit(X_train_main, y_train_main)
    val_preds = model.predict(X_val)
    acc = accuracy_score(y_val, val_preds)
    model_weights[name] = acc
    print(f"{name} validation accuracy: {acc:.4f}")



RandomForest validation accuracy: 0.9930
AdaBoost validation accuracy: 0.9655
NaiveBayes validation accuracy: 0.8357
DecisionTree validation accuracy: 0.9768
SVM validation accuracy: 0.9801


In [35]:
# normalizing weights

totla_weight = sum(model_weights.values())
normalized_weights = {k:v / totla_weight for k,v in model_weights.items()}


In [36]:
# getting predictions from all models
test_probs =[]
for name, model in models.items():
    prob = model.predict_proba(X_test)[:,1]
    weighted_prob = prob * normalized_weights[name]
    test_probs.append(weighted_prob)

In [38]:
# computing the fused score and predictions
fused_score = np.sum(test_probs, axis=0)
final_preds_aab = (fused_score > 0.5).astype(int)  

In [39]:
# Evaluations
precision_abb = precision_score(y_test, final_preds)
recall_abb = recall_score(y_test, final_preds)
f1_abb = f1_score(y_test, final_preds)
cm_abb = confusion_matrix(y_test, final_preds)

print("📊 AAB Fusion Results:")
print(f"Precision: {precision_abb:.4f}")
print(f"Recall (TPR): {recall_abb:.4f}")
print(f"F1-score: {f1_abb:.4f}")
print("Confusion Matrix:")
print(cm_abb)

📊 AAB Fusion Results:
Precision: 0.9795
Recall (TPR): 0.9838
F1-score: 0.9817
Confusion Matrix:
[[2375   28]
 [  22 1340]]


In [40]:
# RAPC (Rank Aggregation with Probabilistic Confidence)
rapc_probs = []
# getting predictions probabilities again

for name, model in models.items():
    prob = model.predict_proba(X_test)[:, 1]
    rapc_probs.append(prob)


rapc_probs = np.column_stack(rapc_probs)

In [41]:
# ranking each classifier's output per sample
# for each row, converting to ranks (0= lowest_prob, n-1 = highest_prob)
rank_matix = rapc_probs.argsort(axis=1).argsort(axis=1)

In [42]:
# average rank across classifiers
avg_rank = rank_matix.mean(axis=1)

In [43]:
# prediction using a threshold median
threshold = np.median(avg_rank)
final_preds_rapc = (avg_rank <= threshold).astype(int)  # lower rank = higher confidence


In [44]:
# Evaluating RAPC
precision_rapc = precision_score(y_test,final_preds_rapc)
recall_rapc = recall_score(y_test,final_preds_rapc)
f1_rapc = f1_score(y_test, final_preds_rapc)
cm_rapc = confusion_matrix(y_test, final_preds_rapc)


print("📊 RAPC Fusion Results:")
print(f"Precision: {precision:.4f}")
print(f"Recall (TPR): {recall:.4f}")
print(f"F1-score: {f1:.4f}")
print("Confusion Matrix:")
print(cm)

📊 RAPC Fusion Results:
Precision: 0.9795
Recall (TPR): 0.9838
F1-score: 0.9817
Confusion Matrix:
[[2375   28]
 [  22 1340]]


In [ ]:
import joblib
import os

